# 65 · Kiểm **KHU TRÚ** attention — cổng cho hướng cascade

## Câu hỏi

ML-Decoder cho mỗi nhãn một phân bố attention riêng trên `1369` patch. Hướng **cascade**
(lượt thô định vị → crop ở độ phân giải gốc → lượt hai) chỉ có cơ sở nếu attention đó
**thật sự khu trú**: giữ một phần nhỏ patch được attend mạnh nhất mà điểm đọc gần như không đổi.

## Phép kiểm (không cần mask/bbox)

Với từng cặp `(ảnh, nhãn)`: lấy attention của lượt **đầy đủ**, giữ `top-k%` patch, chặn phần
còn lại ngay tại cross-attention (`-inf` trước softmax), đo lại AP trên **toàn bộ val**.

```
k = 5% giu >= 90% AP cua luot day du  ->  KHU TRU   -> cascade co co so
k phai len 50% moi giu 90%            ->  TAN DEU   -> cascade KHONG co gi de crop
```

Đo **theo tier** — khu trú ở head (tổn thương lớn) nhưng tán đều ở tail tự nó là phát hiện.

## Cần gì

- checkpoint `stage4_nb53/r3/ckpt/s42_resume.pt` (mô hình `t1_prior_both` đã chốt, epoch 10)
- ảnh 518 staged local (C1 tự kéo nếu thiếu — lâu nhất ~40 phút nếu cache trống)

**Phần cứng:** GPU (T4 đủ), 1 lượt eval val 39,293 ảnh × 2 forward backbone ≈ **~25–35 phút**;
các lượt mask chỉ chạy lại decoder nên gần như miễn phí. **KHÔNG huấn luyện gì.**

Notebook này **CHỈ ĐỌC** checkpoint/dữ liệu của `53` và ghi kết quả vào `stage4_nb65`.


## C0 · Config

In [1]:
import os, json, math, time, random, tarfile, io as _io, re, glob, datetime
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = './planB_out'

OUT_DIR = os.path.join(DATA_DIR, 'cxrlt_2024')
EMB_DIR = os.path.join(OUT_DIR, 'embeddings')
RUN_TAG = 'r3'      # doi vi Tier1Img da doi (LayerScale) — DUNG resume qua tag cu      # r1 = ban `const` cu (W/b/joint cap phat ma khong dung). DUNG resume qua r1.
RES53   = os.path.join(OUT_DIR, 'stage4_nb53', RUN_TAG)
CKPT    = os.path.join(RES53, 'ckpt')
P_OOF   = os.path.join(OUT_DIR, 'stage4_nb52a', 'xgb_oof.npy')
# TACH theo notebook: hai tien trinh cung `append` mot file qua Drive FUSE se nuot dong
# cua nhau. 53 va 57 chay song song.
P_TOUCH = os.path.join(OUT_DIR, '_test_touches_nb53.jsonl')     # nhat ky MOI lan cham test

# KHONG bao gio va `builtins.open` (de quy vo han + chet worker DataLoader).
# 65 CHI DOC RES53/CKPT. KHONG makedirs len thu muc DA ton tai (bai hoc thu muc trung ten
# tren DriveFS: makedirs lap tren listing cu tao thu muc TRUNG TEN).
RES65 = os.path.join(OUT_DIR, 'stage4_nb65')
if not os.path.isdir(RES65): os.makedirs(RES65)      # MOT lan duy nhat
def ensure_res(): return RES65
LOCAL = '/content/cxrlt_518'

BB_RUN = 'a2_E3'
SEEDS  = (42, 43)             # 2 seed: vua doc duoc bien do, VUA nuoi ensemble cua 48
SEED, SIZE, EPOCHS = 42, 518, 10
BATCH, ACCUM = 64, 4          # HAI anh moi mau (phim truoc) -> giu EFF 256
LR, WD, WARMUP, CLIP = 1e-3, 1e-2, 0.05, 5.0
W_FRONTAL, W_LATERAL = 0.7, 0.3
NW = min(8, (os.cpu_count() or 2))
D_MLD, D_ETOK, EHR_CLIP = 512, 256, 10.0

DO_TEST = False               # 65 khong bao gio cham test

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
assert dev == 'cuda', 'KHONG thay GPU -> Runtime > Change runtime type > GPU roi chay lai tu C0.'
AMP = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f'thiet bi: {torch.cuda.get_device_name(0)} | '
      f'VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB | AMP {AMP}')

BB_SRC  = os.path.join(OUT_DIR, 'stage4_nb38a', BB_RUN, 'backbone_ft.pt')
P_MICRO = os.path.join(OUT_DIR, 'cxrlt_micro_full.parquet')
P_VOCAB = os.path.join(OUT_DIR, 'cxrlt_micro_vocab.json')
for p, hd in ((BB_SRC, '38a'), (P_MICRO, '29c'), (P_VOCAB, '29c'), (P_OOF, '52a')):
    assert os.path.exists(p), f'khong thay {p} -> chay {hd} truoc'

INV   = json.load(open(os.path.join(OUT_DIR, 'cxrlt_inventory.json')))
LAB40 = INV['labels_task1']; C = len(LAB40)
TIER  = {t: np.array([LAB40.index(l) for l in INV['tier'][t]]) for t in ('head','medium','tail')}

ORDER = pd.read_parquet(os.path.join(EMB_DIR, 'emb_index.parquet'))
Y = (pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_index.parquet'), columns=['dicom_id'] + LAB40)
       .set_index('dicom_id').loc[ORDER.dicom_id].reset_index())
assert (Y.dicom_id.to_numpy() == ORDER.dicom_id.to_numpy()).all()
YM = Y[LAB40].to_numpy('float32')

P_VIEW = os.path.join(OUT_DIR, 'cxrlt_view_resolved.parquet')
V_ = ((pd.read_parquet(P_VIEW)[['dicom_id','view_group','use']]
         .set_index('dicom_id').loc[ORDER.dicom_id].reset_index()) if os.path.exists(P_VIEW)
      else ORDER[['dicom_id','view_group']].assign(use=lambda d: d.view_group != 'unknown'))

SPLIT = ORDER.split.to_numpy(); SUB = ORDER.subject_id.to_numpy()
TR, VA = SPLIT == 'train', SPLIT == 'val'; TE = ~(TR | VA)
_rs = np.random.default_rng(SEED); _subs = np.unique(SUB[TR])
_sel = _rs.choice(_subs, size=int(round(0.15 * len(_subs))), replace=False)
SL = TR & np.isin(SUB, _sel); TF_ = TR & ~SL     # CUNG cach cat nhu 35/37/38*/44*/45/46/52*
assert not (set(SUB[SL]) & set(SUB[TF_])) and not (set(SUB[TR]) & set(SUB[VA]))
I_TF, I_SL, I_VA, I_TE = (np.where(m)[0] for m in (TF_, SL, VA, TE))
print(f'{len(ORDER):,} anh | khop {len(I_TF):,} | chon {len(I_SL):,} | val {len(I_VA):,} | '
      f'test {len(I_TE):,}')

# --- TIER theo dinh nghia CHINH THUC cua giai (paper §3.3: prev tren TRAIN) ---
# 67 phat hien ban cu cua ta (nb28, chia theo SO CA) ra 13/16/11, con giai chia theo TI LE
# ra 9/14/16. Cung mot dai luong ma bao cao tren hai thang thi khong so duoc voi bang xep hang.
TIER_CU = TIER
_prev = YM[TR].mean(0)
TIER = {'head':   np.where(_prev > 0.10)[0],
        'medium': np.where((_prev >= 0.01) & (_prev <= 0.10))[0],
        'tail':   np.where(_prev < 0.01)[0]}
print('TIER = dinh nghia CHINH THUC ' + str({k: len(v) for k, v in TIER.items()})
      + ' (paper: 9/14/16, tong 39 — thieu 1, chac loi danh may)')
print('TIER_CU (nb28, chia theo so ca) ' + str({k: len(v) for k, v in TIER_CU.items()})
      + '  — giu lai de doi chieu ghi chep cu')
print('!! So tren tier CU trong JOURNAL truoc 2026-09-08 KHONG so duoc voi so tren tier moi.')
print(f'backbone `ft` = 38a/{BB_RUN} | seeds {SEEDS} | E={EPOCHS} | BATCH {BATCH}x{ACCUM}')
print('TEST: TAT — 65 khong bao gio cham test.')

## C1 · EHR + túi vi khuẩn + đặc trưng XGB + **phim chụp trước** — verbatim `45`/`52b`

In [2]:
SPEC = json.load(open(os.path.join(OUT_DIR, 'cxrlt_ehr_spec.json')))
E_DF = pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_ehr.parquet'))
_sid = ORDER.study_id.to_numpy()
def _mat(cols):
    m = (E_DF.set_index('study_id')[cols].reindex(_sid)
             .apply(pd.to_numeric, errors='coerce').to_numpy('float32'))
    return np.nan_to_num(m, nan=0.0)

COLS = [c for c in SPEC['Q_FULL'] if c in E_DF.columns]
_m = _mat(COLS)
# Cot NHI PHAN thi KHONG z-score: p = 1/219,785 -> gia tri chuan hoa 468; `ColTok` dua thang x_j
# vao token nen score ~ M^2 -> softmax bao hoa -> NaN.
_b = np.array([np.isin(np.unique(_m[:, j]), (0.0, 1.0)).all() for j in range(_m.shape[1])])
_mu = _m[TF_].mean(0, keepdims=True); _sd = _m[TF_].std(0, keepdims=True) + 1e-6
_mu[:, _b] = 0.0; _sd[:, _b] = 1.0
E_ALL = np.clip((_m - _mu) / _sd, -EHR_CLIP, EHR_CLIP).astype('float32')
_keep = [i for i, c in enumerate(COLS) if not re.match(r'^micro_org_', c)]
assert len(COLS) - len(_keep) == 30, f'bo {len(COLS)-len(_keep)} cot micro_org_*, phai la 30'
COLS = [COLS[i] for i in _keep]; EHR_COL = E_ALL[:, _keep]

_MIC = pd.read_parquet(P_MICRO).set_index('study_id')
_VOC = json.load(open(P_VOCAB)); V_MIC = int(_VOC['V']) + 1; PAD_ID = V_MIC - 1
_mic = _MIC.reindex(_sid)
_ids = _mic.micro_org_ids.apply(lambda v: list(v) if isinstance(v, (list, np.ndarray)) else [])
_cnt = _mic.micro_org_cnt.apply(lambda v: list(v) if isinstance(v, (list, np.ndarray)) else [])
MIC_L = max(1, int(_ids.map(len).max())); assert MIC_L <= 64
MIDS = np.full((len(ORDER), MIC_L), PAD_ID, 'int64'); MCNT = np.zeros((len(ORDER), MIC_L), 'float32')
for r, (a, b) in enumerate(zip(_ids, _cnt)):
    if a: MIDS[r, :len(a)] = a; MCNT[r, :len(a)] = b
assert (MIDS[MCNT == 0] == PAD_ID).all() and (MIDS[MCNT > 0] != PAD_ID).all()

XGF = np.load(P_OOF).astype('float32')
assert XGF.shape == (len(ORDER), C) and np.isfinite(XGF).all(), f'xgb_oof {XGF.shape}'
_l = np.log(np.clip(XGF, 1e-6, 1-1e-6) / (1 - np.clip(XGF, 1e-6, 1-1e-6)))
XGB_F = np.clip((_l - _l[TF_].mean(0, keepdims=True)) / (_l[TF_].std(0, keepdims=True) + 1e-6),
                -EHR_CLIP, EHR_CLIP).astype('float32')
BOTH_F = np.concatenate([EHR_COL, XGB_F], 1)
print(f'EHR_COL {EHR_COL.shape} | XGB_F {XGB_F.shape} | BOTH_F {BOTH_F.shape} | '
      f'tui vi khuan V={V_MIC-1} dai {MIC_L}')
# GHI CHU: hang TEST cua `xgb_oof` la HANG SO prevalence (52a co y KHONG du doan test). Nen
# `t1_prior_xgb` tren TEST chi nhan mot vector khong doi cho moi anh -> con so test cua HAI arm
# `xgb` KHONG doc duoc. C10 se in canh bao va loai chung khoi bang test.

_BASE = pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_base.parquet'))
_BASE['index_time'] = pd.to_datetime(_BASE.index_time)
_B = (_BASE[_BASE.study_id.isin(set(ORDER.study_id.unique()))]
      .sort_values(['subject_id','index_time']).reset_index(drop=True))
_B['prev_study'] = _B.groupby('subject_id', sort=False).study_id.shift(1)
_PV = dict(zip(_B.study_id.to_numpy(), _B.prev_study.to_numpy()))
_rw = {}
for _i, _s in enumerate(_sid): _rw.setdefault(_s, []).append(_i)
PREV_ROW = np.full(len(ORDER), -1, 'int64')
for _i, _s in enumerate(_sid):
    _p2 = _PV.get(_s, np.nan)
    if not (isinstance(_p2, float) and np.isnan(_p2)): PREV_ROW[_i] = _rw[int(_p2)][0]
_hp = PREV_ROW >= 0
print(f'anh co phim truoc: {_hp.sum():,} / {len(ORDER):,} ({_hp.mean():.1%})  (44a do 67.2%)')
assert _hp.mean() > 0.5, 'ty le qua thap -> kiem lai ghep study'
assert (PREV_ROW[_hp] != np.arange(len(ORDER))[_hp]).all(), 'phim truoc TRUNG chinh no'

EHR_COL (377110, 217) | XGB_F (377110, 40) | BOTH_F (377110, 257) | tui vi khuan V=305 dai 10
anh co phim truoc: 253,283 / 377,110 (67.2%)  (44a do 67.2%)


## C2 · Ảnh + backbone `ft` đông cứng + `DS` trả **hai** ảnh

In [3]:
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from transformers import AutoModel, AutoImageProcessor
from torchvision import transforms as T

os.makedirs(LOCAL, exist_ok=True)
_n = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
print(f'anh local: {_n:,} / {len(ORDER):,}')
if _n < len(ORDER) * 0.999:
    SHARD_DIR = os.path.join(DATA_DIR, 'cxr_1022_letterbox_cxrlt')      # tu 28c C0
    assert os.path.isdir(SHARD_DIR), f'khong thay {SHARD_DIR}'
    TARS = sorted(os.path.join(SHARD_DIR, f) for f in os.listdir(SHARD_DIR)
                  if f.endswith('.tar') and f.startswith('lb_'))
    DONE = os.path.join(LOCAL, '_done.json')
    done = set(json.load(open(DONE))) if os.path.exists(DONE) else set()
    def _one(pl):
        nm, blob = pl; dst = os.path.join(LOCAL, os.path.basename(nm))
        if os.path.exists(dst): return
        Image.open(_io.BytesIO(blob)).convert('L').resize((SIZE, SIZE), Image.BICUBIC) \
             .save(dst, 'JPEG', quality=95)
    with ThreadPoolExecutor(8) as pool:
        for tp in tqdm([t for t in TARS if os.path.basename(t) not in done], desc='stage', unit='tar'):
            with open(tp, 'rb') as f: buf = f.read()   # doc BULK; lap tren mount FUSE se TREO
            with tarfile.open(fileobj=_io.BytesIO(buf)) as tar:
                jobs = [(m.name, tar.extractfile(m).read())
                        for m in tar.getmembers() if m.name.endswith('.jpg')]
            list(pool.map(_one, jobs)); del buf, jobs
            done.add(os.path.basename(tp)); json.dump(sorted(done), open(DONE, 'w'))
    _n = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
assert _n >= len(ORDER) * 0.999, f'thieu {len(ORDER)-_n:,} anh'

proc = AutoImageProcessor.from_pretrained('microsoft/rad-dino')
MEAN, STD = proc.image_mean[0], proc.image_std[0]
train_tf = T.Compose([T.RandomResizedCrop(SIZE, scale=(0.85, 1.0), ratio=(0.95, 1.05),
                                          interpolation=T.InterpolationMode.BICUBIC),
                      T.RandomRotation(7), T.ToTensor(), T.Normalize([MEAN]*3, [STD]*3)])
eval_tf  = T.Compose([T.ToTensor(), T.Normalize([MEAN]*3, [STD]*3)])

_ck = torch.load(BB_SRC, map_location='cpu', weights_only=False)
_sd0 = {k: v.clone() for k, v in AutoModel.from_pretrained('microsoft/rad-dino').state_dict().items()}
backbone = AutoModel.from_pretrained('microsoft/rad-dino')
backbone.load_state_dict(_ck['bb'], strict=True)
with torch.no_grad():
    _d = max((_sd0[k] - v).abs().max().item() for k, v in backbone.state_dict().items())
del _sd0
assert _d > 1e-6, f'backbone GIONG HET pretrain (lech {_d:.1e}) -> `ft` khong duoc nap'
backbone = backbone.to(dev).eval()
for p in backbone.parameters(): p.requires_grad_(False)
D_CXR = backbone.config.hidden_size
print(f'BACKBONE `ft`: 38a/{BB_RUN} | lech so voi pretrain {_d:.3f}. CHI lay `bb`.')

class DS(Dataset):
    '''Tra ve CA HAI anh: hien tai va PHIM TRUOC (+ `has`). `ehr_perm` cho phep TRAO NHAN QUA.'''
    def __init__(s, idx, tf, ehr_perm=None): s.idx, s.tf, s.perm = np.asarray(idx), tf, ehr_perm
    def __len__(s): return len(s.idx)
    def _img(s, i):
        im = Image.open(os.path.join(LOCAL, f'{ORDER.dicom_id.iloc[i]}.jpg')).convert('L')
        return s.tf(Image.merge('RGB', (im, im, im)))
    def __getitem__(s, k):
        i = int(s.idx[k]); pr = int(PREV_ROW[i]); x = s._img(i)
        return dict(image=x, image_prev=(s._img(pr) if pr >= 0 else torch.zeros_like(x)),
                    has=torch.tensor(1.0 if pr >= 0 else 0.0), sub=int(SUB[i]), pos=k,
                    y=torch.from_numpy(YM[i]), row=i,
                    erow=(i if s.perm is None else int(s.perm[k])))

def loader(idx, tf, shuffle, bs, perm=None, nw=NW, drop_last=None):
    return DataLoader(DS(idx, tf, perm), batch_size=bs, shuffle=shuffle, num_workers=nw,
                      pin_memory=True, drop_last=shuffle if drop_last is None else drop_last,
                      persistent_workers=nw > 0, prefetch_factor=4 if nw > 0 else None)

@torch.no_grad()
def fwd(img):
    with torch.autocast('cuda', dtype=AMP, enabled=dev == 'cuda'):
        return backbone(pixel_values=img).last_hidden_state[:, 1:].float()

_b0 = next(iter(loader(I_VA[:8], eval_tf, False, 8, nw=0)))
N_TOK = fwd(_b0['image'].to(dev)).shape[1]
assert N_TOK == (SIZE // backbone.config.patch_size) ** 2
print(f'patch {N_TOK} token')

anh local: 0 / 377,110


stage:   0%|          | 0/378 [00:00<?, ?tar/s]

preprocessor_config.json:   0%|          | 0.00/756 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

BACKBONE `ft`: 38a/a2_E3 | lech so voi pretrain 0.021. CHI lay `bb`.
patch 1369 token


## C3 · Kiến trúc — verbatim `45` (+`ColTokBag` của `52b`)

In [4]:
class GroupFC(nn.Module):
    def __init__(s, C, d):
        super().__init__()
        s.w = nn.Parameter(torch.randn(C, d) * d ** -0.5); s.b = nn.Parameter(torch.zeros(C))
    def forward(s, h): return (h * s.w).sum(-1) + s.b

class XAttn(nn.Module):
    def __init__(s, d, nhead=8, ff=2048, drop=0.1):
        super().__init__()
        s.at = nn.MultiheadAttention(d, nhead, dropout=drop, batch_first=True)
        s.n1, s.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        s.ff = nn.Sequential(nn.Linear(d, ff), nn.GELU(), nn.Dropout(drop), nn.Linear(ff, d))
    def forward(s, q, kv, need_w=False):
        a, w = s.at(q, kv, kv, need_weights=need_w, average_attn_weights=True)
        h = s.n1(q + a); return s.n2(h + s.ff(h)), w

class MicroBag(nn.Module):
    '''`const=True`: MOT chi so, khong phai MIC_L ban sao — tui that trung binh 1.7 vi khuan.'''
    def __init__(s, V, d, const=False):
        super().__init__()
        s.const = const
        s.w = nn.Parameter(torch.randn(V, d) * d ** -0.5)
        s.mlp = nn.Sequential(nn.GELU(), nn.Linear(d, d))
    def forward(s, ids, w):
        if s.const:
            ids = torch.zeros(ids.shape[0], 1, dtype=torch.long, device=ids.device)
            w = torch.ones(ids.shape[0], 1, device=ids.device)
        return s.mlp(F.embedding_bag(ids, s.w, per_sample_weights=w, mode='sum'))

class ColTokBag(nn.Module):
    '''d_in token COT (T_j = b_j + x_j*W_j) + 1 token TONG HOP (MLP nhin CA d_in cot) + 1 token
       BAG vi khuan (tuy chon). Chuoi nay CHUA k1 nhu TRUONG HOP RIENG — attention chi can don het
       khoi luong len token tong hop la tai lap duoc k1.'''
    def __init__(s, d_in, d=256, const=False, V=None):
        super().__init__()
        s.W = nn.Parameter(torch.randn(d_in, d) * d ** -0.5)
        s.b = nn.Parameter(torch.zeros(d_in, d))
        s.joint = nn.Sequential(nn.Linear(d_in, d), nn.GELU(), nn.Linear(d, d))
        s.ln = nn.LayerNorm(d); s.const = const
        # `const` cu thay THANG chuoi token bang mot ma tran hang so, khien `W`/`b`/`joint` duoc
        # CAP PHAT nhung KHONG BAO GIO dung -> chung van bi dem vao `NP_` (nen cong "doi chung bao
        # thu" dat mot cach gia tao) trong khi dung luong THUC te thap hon han arm that.
        # Do la dung lop lech ma `56` phoi bay: doi chung ngheo hon ve VAI TRO thi de sup hon, va
        # moi hieu `arm - doi chung` bi thoi len.
        # Nay doi chung chay DUNG bo may do, chi khac DAU VAO la mot vector HANG SO hoc duoc.
        if const: s.c_in = nn.Parameter(torch.zeros(d_in))
        s.mb = MicroBag(V, d, const) if V else None
        if s.mb is not None:
            s.bln = nn.LayerNorm(d); s.cln = nn.LayerNorm(d) if const else nn.Identity()
    def forward(s, e, mids=None, mcnt=None):
        if s.const: e = s.c_in.unsqueeze(0).expand(e.shape[0], -1)   # chi DAU VAO la hang so
        col = s.b.unsqueeze(0) + e.unsqueeze(-1) * s.W.unsqueeze(0)
        base = s.ln(torch.cat([col, s.joint(e).unsqueeze(1)], 1))
        if s.mb is None: return base
        return torch.cat([s.cln(base), s.bln(s.mb(mids, mcnt)).unsqueeze(1)], 1)

class Tier1(nn.Module):
    '''Q = 1369 PATCH, K/V = token EHR. softmax CHI tren token EHR nen EHR khong the bi 1369 patch
       nhan chim (co che that bai cua 06b H6). `pr` zero-init; `ln_a` chuan hoa DAU VAO cua phep
       chieu — ban va 38d, thieu no thi `t1_zero` sup 0.2416 -> 0.0681.'''
    def __init__(s, d, nhead=8, drop=0.1):
        super().__init__()
        s.at = nn.MultiheadAttention(d, nhead, dropout=drop, batch_first=True)
        s.n = nn.LayerNorm(d); s.ln_a = nn.LayerNorm(d); s.pr = nn.Linear(d, d)
        nn.init.zeros_(s.pr.weight); nn.init.zeros_(s.pr.bias)
    def forward(s, patch, etok):
        a, _ = s.at(s.n(patch), etok, etok, need_weights=False)
        return patch + s.pr(s.ln_a(a))

class Tier1Img(nn.Module):
    '''Phim truoc o muc PATCH. Q = 1369 patch HIEN TAI, K/V = 1369 patch PHIM CU, chay o khong gian
       D_CXR (TRUOC `pin`) => MLDecoder khong phai sua gi.
       KHONG tru thang `patch_cur - patch_prior`: hai phim khong duoc dang ky (tu the, do hit vao),
       o `i` co the la dinh phoi o phim nay va xuong don o phim kia. Cross-attention tu tim tuong ung.'''
    def __init__(s, d, nhead=8, drop=0.1, diff=True):
        super().__init__()
        s.at = nn.MultiheadAttention(d, nhead, dropout=drop, batch_first=True)
        s.nq = nn.LayerNorm(d); s.nk = nn.LayerNorm(d); s.ln_a = nn.LayerNorm(d)
        # LAYERSCALE. `pr` zero-init cho dong nhat thuc o buoc 0 nhung KHONG chan duoc toc do
        # lon len: AdamW cap nhat ~lr/buoc/tham so BAT KE gradient nho the nao, nen mot nhanh xuat
        # phat tu 0 lon TUYEN TINH theo buoc x LR. Do `ln_a` chuan hoa `patch - a` (thuong gan 0,
        # phan lon la nhieu) ve chuan don vi, bien do dau ra do TRON VEN `‖pr‖` quyet dinh. Khi no
        # at `patch` thi ML-Decoder doi che do dot ngot -> sup. Da xay ra 3 lan, luon o ep2-4 tuc
        # ngay sau dinh warmup (0.05 x E = nua epoch 1), va `clip` do 0.0% vi gradient VAN binh
        # thuong — van de o BUOC DI, khong o gradient.
        # Nay `pr` khoi tao CHUAN va mot he so LayerScale `1e-4` giu phan du gan 0 luc dau; ca hai
        # deu co gradient that, va he so phai HOC de lon len thay vi lon lên vi quan tinh AdamW.
        s.pr = nn.Linear(d, d)
        s.gamma = nn.Parameter(torch.full((d,), 1e-4))
        s.diff = diff
    def forward(s, patch, prior, has):
        a, _ = s.at(s.nq(patch), s.nk(prior), s.nk(prior), need_weights=False)
        z = s.gamma * s.pr(s.ln_a(patch - a if s.diff else a))
        return patch + z * has.view(-1, 1, 1)      # khong co phim truoc -> mask nhanh di han

class MLDecoder(nn.Module):
    '''Q = C query lop (KHONG hoc, dung theo bai goc). K/V = 1369 patch, co the da bi Tier1 dieu bien.'''
    def __init__(s, C, d_in, d=512, nhead=8, ff=1024, drop=0.1, d_ehr_in=0, t1=False):
        super().__init__()
        s.q = nn.Embedding(C, d); s.q.weight.requires_grad_(False)
        s.pin = nn.Linear(d_in, d); s.img = XAttn(d, nhead, ff, drop)
        s.t1 = Tier1(d, nhead, drop) if t1 else None
        if t1: s.ein = nn.Linear(d_ehr_in, d)
        s.out = GroupFC(C, d)
    def forward(s, patch, etok=None, need_w=False):
        q = s.q.weight.unsqueeze(0).expand(patch.shape[0], -1, -1)
        kv = s.pin(patch)
        if s.t1 is not None: kv = s.t1(kv, s.ein(etok))
        h, w = s.img(q, kv, need_w); return s.out(h), w

class ASL(nn.Module):
    '''Verbatim nb11/12/13/15/35/37/38*/44*/45. gn=4, gp=0 la cau hinh 35 da chot.'''
    def __init__(s, gn=4, gp=0, clip=0.05, eps=1e-8):
        super().__init__(); s.gn, s.gp, s.clip, s.eps = gn, gp, clip, eps
    def forward(s, x, y):
        y = y.float(); xp = torch.sigmoid(x); xn = 1 - xp
        if s.clip: xn = (xn + s.clip).clamp(max=1)
        lo = y*torch.log(xp.clamp(min=s.eps)) + (1-y)*torch.log(xn.clamp(min=s.eps))
        pt = xp*y + xn*(1-y); g = s.gp*y + s.gn*(1-y)
        return -(lo * torch.pow(1-pt, g)).mean()
print('kien truc san sang')

kien truc san sang


## C4 · 7 arm + cổng khoá trước

In [5]:
ARMS = {
  'mldec':             dict(feat=None,   img=False),
  't1_prior':          dict(feat='col',  img=True),
  't1_prior_ehrz':     dict(feat='col',  img=True, const=True),
  't1_prior_imgz':     dict(feat='col',  img=True, shuf=True),
  't1_prior_xgb':      dict(feat='xgb',  img=True, bag=False),   # XGB da tom tat CA vi khuan
  't1_prior_xgb_ehrz': dict(feat='xgb',  img=True, bag=False, const=True),
  't1_prior_both':     dict(feat='both', img=True),
}
FEAT = {'col': EHR_COL, 'xgb': XGB_F, 'both': BOTH_F}

def build_arms(seed=SEED):
    M, E, I = nn.ModuleDict(), nn.ModuleDict(), nn.ModuleDict()
    for tag, cfg in ARMS.items():
        torch.manual_seed(seed)      # gieo lai TRUOC TUNG arm -> khoi tao giong het nhau
        if cfg['feat'] is not None:
            E[tag] = ColTokBag(FEAT[cfg['feat']].shape[1], D_ETOK, cfg.get('const', False),
                               V=(V_MIC if cfg.get('bag', True) else None))
        M[tag] = MLDecoder(C, D_CXR, D_MLD, d_ehr_in=D_ETOK, t1=cfg['feat'] is not None)
        if cfg['img']: I[tag] = Tier1Img(D_CXR)
    return M.to(dev), E.to(dev), I.to(dev)

MOD, ETK, IMG = build_arms()
NP_ = {t: sum(p.numel() for m in ([MOD[t]] + ([ETK[t]] if t in ETK else [])
                                  + ([IMG[t]] if t in IMG else []))
              for p in m.parameters() if p.requires_grad) for t in ARMS}
print(f'{"arm":<20} {"tham so":>12}  {"dac trung":>9}  {"tui":>4}')
for t, c in ARMS.items():
    print(f'  {t:<18} {NP_[t]:>12,}  {str(c["feat"]):>9}  {str(c.get("bag", True)):>4}')

_e = {k: torch.randn(4, v.shape[1], device=dev) for k, v in FEAT.items()}
_mi = torch.zeros(4, MIC_L, dtype=torch.long, device=dev); _mc = torch.zeros(4, MIC_L, device=dev)
_p = torch.randn(4, N_TOK, D_CXR, device=dev); _pp = torch.randn(4, N_TOK, D_CXR, device=dev)
_h1 = torch.ones(4, device=dev); _h0 = torch.zeros(4, device=dev)
tok = lambda t: ETK[t](_e[ARMS[t]['feat']], _mi, _mc)

assert not any(p.requires_grad for p in backbone.parameters()), 'backbone chua dong cung'
assert not MOD['mldec'].q.weight.requires_grad, 'query_embed dang hoc'
with torch.no_grad():
    m = MOD['t1_prior'].eval(); l1, _ = m(_p, tok('t1_prior'))
    _k = m.t1; m.t1 = None; l2, _ = m(_p, tok('t1_prior')); m.t1 = _k
    d1 = (l1 - l2).abs().max().item()
assert d1 < 1e-5, f'Tier1 KHONG zero-init: lech {d1:.2e}'
print(f'\ncong 1: Tier1 zero-init OK (lech {d1:.1e})')
with torch.no_grad():
    for t in IMG:
        # Cong TUONG DOI thay vi tuyet doi: LayerScale cho dong nhat thuc XAP XI (~1e-4), khong
        # bang chinh xac. Va TI LE BIEN DO moi la thu quyet dinh — chinh no vuot 1 la sup.
        _rel = float((IMG[t](_p, _pp, _h1) - _p).norm() / (_p.norm() + 1e-9))
        assert _rel < 1e-3, f'{t}: Tier1Img nhieu {_rel:.2e} luc khoi tao, phai < 1e-3'
print(f'cong 2: Tier1Img zero-init OK cho {len(IMG)} arm')
with torch.no_grad():
    _ti = IMG['t1_prior']; _ti.pr.weight.normal_(0, 0.02)
    _r = float((_ti(_p, _pp*100.0, _h1) - _p).abs().max() / ((_ti(_p, _pp, _h1) - _p).abs().max()+1e-9))
    assert (_ti(_p, _pp, _h0) - _p).abs().max() < 1e-6, 'has=0 KHONG cat duoc nhanh phim truoc'
    nn.init.zeros_(_ti.pr.weight)
assert _r < 3.0, f'phong dai K/V 100x -> bien do ra tang {_r:.1f}x, `ln_a` khong chan duoc'
print(f'cong 3: `ln_a` chan bien do OK ({_r:.2f}x) | has=0 cat sach nhanh OK')
for a, b in (('t1_prior','t1_prior_ehrz'), ('t1_prior_xgb','t1_prior_xgb_ehrz')):
    assert NP_[b] >= NP_[a], f'{b} IT tham so hon {a} -> doi chung khong con bao thu'
    print(f'cong 4: {b} {NP_[b]:,} >= {a} {NP_[a]:,} (+{NP_[b]-NP_[a]:,}) -> `{a} - {b} > 0` BAO THU')
assert NP_['t1_prior'] == NP_['t1_prior_imgz'], 'imgz phai khop tham so TUYET DOI voi t1_prior'
print(f'cong 5: t1_prior = t1_prior_imgz ({NP_["t1_prior"]:,}) — khop TUYET DOI')

arm                       tham so  dac trung   tui
  mldec                 2,517,032       None  True
  t1_prior              7,300,136        col  True
  t1_prior_ehrz         7,300,865        col  True
  t1_prior_imgz         7,300,136        col  True
  t1_prior_xgb          7,019,560        xgb  False
  t1_prior_xgb_ehrz     7,019,600        xgb  False
  t1_prior_both         7,330,856       both  True

cong 1: Tier1 zero-init OK (lech 0.0e+00)
cong 2: Tier1Img zero-init OK cho 6 arm
cong 3: `ln_a` chan bien do OK (1.00x) | has=0 cat sach nhanh OK
cong 4: t1_prior_ehrz 7,300,865 >= t1_prior 7,300,136 (+729) -> `t1_prior - t1_prior_ehrz > 0` BAO THU
cong 4: t1_prior_xgb_ehrz 7,019,600 >= t1_prior_xgb 7,019,560 (+40) -> `t1_prior_xgb - t1_prior_xgb_ehrz > 0` BAO THU
cong 5: t1_prior = t1_prior_imgz (7,300,136) — khop TUYET DOI


## C5 · Chấm điểm — verbatim `45`

In [6]:
from sklearn.metrics import average_precision_score
def ap_cols(Yc, Sc):
    o = np.full(Yc.shape[1], np.nan)
    for j in range(Yc.shape[1]):
        y, s = Yc[:, j], Sc[:, j]; n1 = y.sum()
        if n1 == 0 or n1 == len(y): continue
        r = np.argsort(-s, kind='stable'); yy, ss = y[r], s[r]
        tp, fp = np.cumsum(yy), np.cumsum(1 - yy)
        last = np.r_[np.diff(ss) != 0, True]; tp, fp = tp[last], fp[last]
        pr, rc = tp / (tp + fp), tp / n1
        o[j] = float((np.diff(np.r_[0.0, rc]) * pr).sum())
    return o
tiers = lambda ap: {t: float(np.nanmean(ap[i])) for t, i in TIER.items()}
_rs6 = np.random.default_rng(6); _yy = (_rs6.random((3000, C)) < 0.2).astype('float32')
_ss6 = _rs6.random((3000, C)).astype('float32')
_dd6 = float(np.nanmax(np.abs(ap_cols(_yy, _ss6) -
        np.array([average_precision_score(_yy[:, j], _ss6[:, j]) for j in range(C)]))))
assert _dd6 < 1e-6, f'ap_cols lech sklearn {_dd6:.2e}'
print(f'cong 6: ap_cols khop sklearn toi {_dd6:.2e}')

cong 6: ap_cols khop sklearn toi 1.69e-08


## Phân tích: giữ top-k% patch, đo AP còn lại

Một lượt qua val: backbone forward **một lần mỗi batch** (phần đắt), lượt đầy đủ lấy cả
attention `w [B, C, 1369]`, rồi **mỗi k** chỉ chạy lại tầng cross-attention với mask —
gần như miễn phí.

Kèm hai chẩn đoán độc lập với mask:
- **độ tập trung**: tổng khối lượng attention nằm trong top-5% patch, theo nhãn;
- ba nhãn tail đã biết bị `cnn_ms` phá được in riêng.


In [7]:
# ==== KHU TRU ATTENTION: giu top-k% patch, chan phan con lai, do lai AP ====
TAG = 't1_prior_both'
KS  = (0.01, 0.02, 0.05, 0.10, 0.20, 0.50)
NGUONG_GIU, K_CONG = 0.90, 0.05    # cong khoa truoc: k=5% giu >=90% AP -> KHU TRU
SEED_CK       = 42
CHON_TREN_LAT = True               # quet them tren LAT CHON de chon `k` cho HOP LE
                                   # (chon k tren val roi bao cao tren val = tu lua)

FEAT_T = {k: torch.from_numpy(v) for k, v in FEAT.items()}
MIDS_T, MCNT_T = torch.from_numpy(MIDS), torch.from_numpy(MCNT)

# ---- nguon trong so: uu tien 64/r2a (tung epoch), lui ve resume cua 53 ----
# Kien truc `tpb` cua 64 == `t1_prior_both` cua 53 (cung BOTH_F 257 cot, cung ColTokBag/
# MLDecoder/Tier1Img), chi khac TEN arm -> nap theo TUNG NHANH, khong nap ca ModuleDict.
NG = []
_d64 = os.path.join(OUT_DIR, 'stage4_nb64', 'r2a', 'ckpt')
if os.path.isdir(_d64):
    for f in sorted(os.listdir(_d64)):
        _m2 = re.match(rf's{SEED_CK}_w_ep(\d+)\.pt$', f)
        if _m2: NG.append((int(_m2.group(1)), os.path.join(_d64, f), 'tpb', '64/r2a'))
_p53 = os.path.join(CKPT, f's{SEED_CK}_resume.pt')
if os.path.exists(_p53):
    try:
        NG.append((int(torch.load(_p53, map_location='cpu', weights_only=False)['ep']) + 1,
                   _p53, 't1_prior_both', '53/r3'))
    except Exception as _e0:
        print(f'doc {_p53} loi: {type(_e0).__name__}')
assert NG, 'khong thay checkpoint nao (64/r2a lan 53/r3)'
NG.sort(key=lambda z: (z[3] == '53/r3', -z[0]))      # 64 truoc, roi epoch cao nhat
EP_CK, _ck_p, TAG_CK, _NGUON = NG[0]

_r = torch.load(_ck_p, map_location=dev, weights_only=False)
_nhanh = lambda sd, tg: {k[len(tg)+1:]: v for k, v in sd.items() if k.startswith(tg + '.')}
for _md, _key in ((MOD, 'mod'), (ETK, 'etk'), (IMG, 'img')):
    _w = _nhanh(_r[_key], TAG_CK)
    assert _w, f'khong thay nhanh `{TAG_CK}` trong `{_key}` cua {_ck_p}'
    _md[TAG].load_state_dict(_w)
print(f'nap {_NGUON} seed {SEED_CK} epoch {EP_CK}/{EPOCHS}  (arm `{TAG_CK}` -> `{TAG}`)')
if EP_CK < EPOCHS:
    print(f'  !! CHI epoch {EP_CK} — moi con so duoi la SO BO. Trong so epoch cuoi cua 53/r3 da')
    print(f'     mat (ghi de qua FUSE that bai); do lai sau khi 64 chay xong.')

m = MOD[TAG].eval(); ETK[TAG].eval(); IMG[TAG].eval()
NHEAD = m.img.at.num_heads

@torch.no_grad()
def _logit_mask(kv, w, kfrac):
    """Chay lai TANG CUOI voi attention bi chan ngoai top-k. `kv` da qua pin+Tier1 (tinh 1 lan)."""
    B, Cq, N = w.shape
    kq = max(1, int(round(kfrac * N)))
    allow = torch.zeros_like(w, dtype=torch.bool).scatter_(-1, w.topk(kq, -1).indices, True)
    am = torch.zeros_like(w).masked_fill_(~allow, float('-inf')).repeat_interleave(NHEAD, 0)
    q = m.q.weight.unsqueeze(0).expand(B, -1, -1)
    a, _ = m.img.at(q, kv, kv, need_weights=False, attn_mask=am)
    h = m.img.n1(q + a); h = m.img.n2(h + m.img.ff(h))
    return m.out(h)

@torch.no_grad()
def _quet(IDX, ten):
    PF = np.zeros((len(IDX), C), 'float32')
    PK = {k: np.zeros((len(IDX), C), 'float32') for k in KS}
    T5 = np.zeros(C, 'float64'); nb_ = 0
    for b in tqdm(loader(IDX, eval_tf, False, 48, None, nw=0, drop_last=False), desc=ten):
        pt = fwd(b['image'].to(dev, non_blocking=True))
        pp = fwd(b['image_prev'].to(dev, non_blocking=True))
        erow = b['erow']
        p = IMG[TAG](pt, pp, b['has'].to(dev))
        kv = m.pin(p)
        kv = m.t1(kv, m.ein(ETK[TAG](FEAT_T['both'][erow].to(dev),
                                     MIDS_T[erow].to(dev), MCNT_T[erow].to(dev))))
        q = m.q.weight.unsqueeze(0).expand(kv.shape[0], -1, -1)
        h, w = m.img(q, kv, need_w=True)                   # w [B, C, N], da TB qua head
        pos = b['pos'].numpy()
        PF[pos] = torch.sigmoid(m.out(h).float()).cpu().numpy()
        for k in KS: PK[k][pos] = torch.sigmoid(_logit_mask(kv, w, k).float()).cpu().numpy()
        k5 = max(1, int(round(0.05 * w.shape[-1])))
        T5 += w.topk(k5, -1).values.sum(-1).mean(0).double().cpu().numpy(); nb_ += 1
    assert np.abs(PF).sum(1).min() > 0, f'{ten}: co hang khong duoc ghi -> `pos` sai'
    return PF, PK, T5 / max(nb_, 1)

PF, PK, TOP5 = _quet(I_VA, 'val')
apF = ap_cols(YM[VA], PF)
print(f'\nluot DAY DU tren val: macro {np.nanmean(apF):.4f}')

def _bang(Y_, PF_, PK_, ten):
    apF_ = ap_cols(Y_, PF_)
    print(f'\n[{ten}]  ti le AP_k / AP_day_du, TB theo tier')
    print(f'{"k giu":>7} | ' + ' | '.join(f'{t:>9}' for t in ('head', 'medium', 'tail')))
    G = {}
    for k in KS:
        apk = ap_cols(Y_, PK_[k])
        G[k] = {t: float(np.nanmean(apk[i]) / max(np.nanmean(apF_[i]), 1e-9))
                for t, i in TIER.items()}
        print(f'{k:>6.0%} | ' + ' | '.join(f'{G[k][t]:>9.3f}' for t in ('head','medium','tail')))
    print(f'{"100%":>7} | ' + ' | '.join(f'{1.0:>9.3f}' for _ in range(3)))
    return G, apF_

GIU, _ = _bang(YM[VA], PF, PK, 'VAL')

print(f'\n=== CONG (khoa truoc): k = {K_CONG:.0%} giu >= {NGUONG_GIU:.0%} AP ===')
for t in ('head', 'medium', 'tail'):
    print(f'  {t:<8} {GIU[K_CONG][t]:.3f}  ->  '
          + ('KHU TRU' if GIU[K_CONG][t] >= NGUONG_GIU else 'TAN DEU'))

print('\nDO TAP TRUNG (khoi luong attention trong top-5% patch; deu tuyet doi = 0.05):')
for t, i in TIER.items(): print(f'  {t:<8} {float(TOP5[i].mean()):.3f}')

_QT = ('Azygos Lobe', 'Hydropneumothorax', 'Pneumomediastinum')
_ap5 = ap_cols(YM[VA], PK[K_CONG])
print('\nba nhan tail (AP day du | giu o k=5% | tap trung):')
for lb in _QT:
    j = LAB40.index(lb)
    print(f'  {lb:<20} {apF[j]:.4f} | {_ap5[j]/max(apF[j],1e-9):.3f} | {TOP5[j]:.3f}')

# ---- chon `k` tren LAT CHON, doc ket qua tren VAL ----
K_SAO = None
if CHON_TREN_LAT:
    PFs, PKs, _ = _quet(I_SL, 'lat chon')
    GIU_SL, _ = _bang(YM[SL], PFs, PKs, 'LAT CHON')
    K_SAO = max(KS, key=lambda k: GIU_SL[k]['tail'])
    print(f'\n=== CHON TREN LAT CHON -> k* = {K_SAO:.0%} '
          f'(tail lat chon {GIU_SL[K_SAO]["tail"]:.3f}) ===')
    apk = ap_cols(YM[VA], PK[K_SAO])
    print(f'{"":<10}{"day du":>10}{"che ngoai k*":>14}{"delta":>10}')
    for t, i in TIER.items():
        a0, a1 = float(np.nanmean(apF[i])), float(np.nanmean(apk[i]))
        print(f'  {t:<8}{a0:>10.4f}{a1:>14.4f}{a1-a0:>+10.4f}')
    a0, a1 = float(np.nanmean(apF)), float(np.nanmean(apk))
    print(f'  {"macro":<8}{a0:>10.4f}{a1:>14.4f}{a1-a0:>+10.4f}')
    print('  k* chon tren LAT CHON, so tren VAL -> doc duoc. Chua cham test.')

_out = dict(seed=SEED_CK, ep=EP_CK, nguon=_NGUON, tag=TAG, ks=list(KS),
            giu={f'{k:.2f}': GIU[k] for k in KS}, k_sao=K_SAO,
            top5={t: float(TOP5[i].mean()) for t, i in TIER.items()},
            macro_day_du=float(np.nanmean(apF)))
_hs = f's{SEED_CK}_{_NGUON.replace("/", "")}_ep{EP_CK}'
json.dump(_out, open(os.path.join(RES65, f'khu_tru_{_hs}.json'), 'w'), indent=1)
np.save(os.path.join(RES65, f'ap_day_du_{_hs}.npy'), apF)
for k in KS: np.save(os.path.join(RES65, f'ap_k{int(k*100):03d}_{_hs}.npy'), ap_cols(YM[VA], PK[k]))
print(f'\nda luu vao {RES65} (hau to {_hs})')
